# Baseline Model — Logistic Regression (Dataset v2)

## Mục đích
Sử dụng **Logistic Regression** làm **mô hình cơ sở (Baseline)** để:
1. Đánh giá nhanh khả năng phân tách của tập đặc trưng 379 features (Rule v2)
2. **Phát hiện Data Leakage**: Nếu AUC-ROC > 0.99 → rất có khả năng features chứa thông tin rò rỉ từ tương lai
3. Thiết lập ngưỡng hiệu năng tối thiểu trước khi đầu tư vào mô hình nâng cao (LightGBM, XGBoost)

## Quy tắc nhãn Churn v2
- **Rule 1** — Đóng tài khoản trong vòng 30 ngày tới
- **Rule 2** — Hạ gói xuống Free + Không có hoạt động trong 30 ngày tới
- **Rule 3** — Đang ở gói Free sẵn + Không có hoạt động trong 30 ngày tới

## Tiêu chí đánh giá Data Leakage

| AUC-ROC trên Test | Đánh giá | Hành động |
|:---|:---|:---|
| **> 0.99** | 🔴 **Nghi ngờ Data Leakage nghiêm trọng** | Dừng lại, kiểm tra từng feature |
| **0.95 – 0.99** | 🟡 **Cần kiểm tra kỹ** | Xem top features, loại bỏ nghi vấn |
| **0.80 – 0.95** | 🟢 **Bình thường** | Tiến hành mô hình nâng cao |
| **< 0.80** | ⚪ **Yếu** | Cần bổ sung features hoặc kiểm tra nhãn |

## 1. Nạp thư viện

In [1]:
import warnings
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve,
    precision_recall_curve, confusion_matrix,
    classification_report, f1_score, brier_score_loss,
)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings("ignore")
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

SEED = 42
np.random.seed(SEED)

print(f"Python  : {sys.version.split()[0]}")
print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")
print(f"sklearn : {__import__('sklearn').__version__}")

Python  : 3.11.9
pandas  : 3.0.5
numpy   : 2.4.6
sklearn : 1.9.0


## 2. Đọc dữ liệu Dataset v2

In [2]:
data_path = Path("output/churn_temporal_dataset_v2.parquet")
if not data_path.exists():
    raise FileNotFoundError(f"Không tìm thấy file: {data_path}")

df = pd.read_parquet(data_path)
df["snapshot_date"] = pd.to_datetime(df["snapshot_date"])

print(f"Kích thước dữ liệu: {df.shape[0]:,} dòng × {df.shape[1]} cột")
print(f"Phạm vi thời gian : {df['snapshot_date'].min().date()} → {df['snapshot_date'].max().date()}")
print(f"Số lượng KH duy nhất: {df['customer_id'].nunique():,}")
print()

# Phân phối nhãn
churn_counts = df["churn_next_30d"].value_counts()
print("Phân phối nhãn Churn v2:")
print(f"  Non-churn (0): {churn_counts.get(0, 0):>8,} ({churn_counts.get(0,0)/len(df):.2%})")
print(f"  Churn     (1): {churn_counts.get(1, 0):>8,} ({churn_counts.get(1,0)/len(df):.2%})")

# Phân tích theo rule
if "rule1_closed" in df.columns:
    print(f"\nChi tiết theo Rule v2:")
    print(f"  Rule 1 (Đóng TK)         : {df['rule1_closed'].sum():,}")
    print(f"  Rule 2 (Hạ gói + Không HĐ): {df['rule2_downgrade_to_free_inactive'].sum():,}")
    print(f"  Rule 3 (Free + Không HĐ)  : {df['rule3_free_at_snapshot_inactive'].sum():,}")

Kích thước dữ liệu: 185,160 dòng × 387 cột
Phạm vi thời gian : 2023-08-01 → 2026-08-01
Số lượng KH duy nhất: 10,002

Phân phối nhãn Churn v2:
  Non-churn (0):  141,617 (76.48%)
  Churn     (1):   43,543 (23.52%)

Chi tiết theo Rule v2:
  Rule 1 (Đóng TK)         : 683
  Rule 2 (Hạ gói + Không HĐ): 180
  Rule 3 (Free + Không HĐ)  : 42,832


## 3. Xác định tập đặc trưng & Loại bỏ cột Meta/Nhãn

**Quan trọng:** Các cột sau phải bị loại bỏ khỏi tập đặc trưng đầu vào:
- `customer_id`, `snapshot_date` — Metadata định danh
- `churn_next_30d` — Nhãn mục tiêu (Target)
- `rule1_closed`, `rule2_downgrade_to_free_inactive`, `rule3_free_at_snapshot_inactive` — Thành phần cấu tạo nhãn (nếu dùng → data leakage 100%)
- `churn_reason`, `tier_at_snapshot`, `closed_date`, `label_complete` — Thông tin liên quan nhãn
- Bất kỳ cột nào bắt đầu bằng `future_` hoặc kết thúc bằng `_future`

In [3]:
# Cột meta/nhãn — TUYỆT ĐỐI KHÔNG được dùng làm feature
META_COLS = {
    "customer_id", "snapshot_date", "churn_next_30d",
    "rule1_closed", "rule2_downgrade_to_free_inactive",
    "rule3_free_at_snapshot_inactive",
    "churn_reason", "tier_at_snapshot", "closed_date", "label_complete",
}

# Lọc đặc trưng — loại bỏ meta và cột liên quan tương lai
feature_cols = [
    c for c in df.columns
    if c not in META_COLS
    and not c.startswith("future_")
    and not c.endswith("_future")
]

# Kiểm tra rò rỉ dữ liệu cơ bản
leaked_candidates = [c for c in feature_cols if any(kw in c.lower() for kw in ["rule", "future", "closed", "churn"])]
if leaked_candidates:
    print(f"⚠️ CẢNH BÁO: Phát hiện {len(leaked_candidates)} cột nghi ngờ rò rỉ:")
    for c in leaked_candidates:
        print(f"    - {c}")
    print("  → Loại bỏ các cột này khỏi tập feature...")
    feature_cols = [c for c in feature_cols if c not in leaked_candidates]

print(f"Số lượng đặc trưng đầu vào: {len(feature_cols)}")
print(f"Cột nhãn mục tiêu: churn_next_30d")

# Hiển thị 10 feature đầu tiên
print(f"\n10 đặc trưng đầu tiên:")
for i, col in enumerate(feature_cols[:10], 1):
    print(f"  {i:2d}. {col} (dtype: {df[col].dtype})")

Số lượng đặc trưng đầu vào: 379
Cột nhãn mục tiêu: churn_next_30d

10 đặc trưng đầu tiên:
   1. orders (dtype: float64)
   2. completed_orders (dtype: float64)
   3. spend (dtype: float32)
   4. usage (dtype: float64)
   5. active_days (dtype: float64)
   6. payment_count (dtype: float64)
   7. payment_success (dtype: float64)
   8. payment_failure (dtype: float64)
   9. support_ticket (dtype: float64)
  10. csat (dtype: float64)


## 4. Phân chia dữ liệu theo dòng thời gian (Chronological Split)

Phân chia nghiêm ngặt theo thứ tự thời gian để mô phỏng thực tế triển khai:

| Tập | Khoảng thời gian | Mục đích |
|:---|:---|:---|
| **Train** | 2024-09 → 2025-08 | Huấn luyện mô hình |
| **Validation** | 2025-09 → 2026-02 | Chọn threshold, hiệu chuẩn |
| **Test** | 2026-03 → 2026-06 | Đánh giá cuối cùng & phát hiện leakage |

In [4]:
TRAIN_START = pd.Timestamp("2024-09-01")
TRAIN_END   = pd.Timestamp("2025-08-01")
VAL_START   = pd.Timestamp("2025-09-01")
VAL_END     = pd.Timestamp("2026-02-01")
TEST_START  = pd.Timestamp("2026-03-01")
TEST_END    = pd.Timestamp("2026-06-01")

train_mask = (df["snapshot_date"] >= TRAIN_START) & (df["snapshot_date"] <= TRAIN_END)
val_mask   = (df["snapshot_date"] >= VAL_START)   & (df["snapshot_date"] <= VAL_END)
test_mask  = (df["snapshot_date"] >= TEST_START)  & (df["snapshot_date"] <= TEST_END)

X_train = df.loc[train_mask, feature_cols]
y_train = df.loc[train_mask, "churn_next_30d"].to_numpy()

X_val   = df.loc[val_mask, feature_cols]
y_val   = df.loc[val_mask, "churn_next_30d"].to_numpy()

X_test  = df.loc[test_mask, feature_cols]
y_test  = df.loc[test_mask, "churn_next_30d"].to_numpy()

print("=" * 72)
print(f"  {'Tập':<12} {'Khoảng thời gian':<28} {'Số dòng':>8} {'Churn':>8} {'Tỷ lệ':>8}")
print("-" * 72)
for name, s, e, y in [
    ("Train", TRAIN_START, TRAIN_END, y_train),
    ("Validation", VAL_START, VAL_END, y_val),
    ("Test", TEST_START, TEST_END, y_test),
]:
    n, c = len(y), int(y.sum())
    rate = c / n if n > 0 else 0
    print(f"  {name:<12} {str(s.date()) + ' → ' + str(e.date()):<28} {n:>8,} {c:>8,} {rate:>7.2%}")
print("=" * 72)

  Tập          Khoảng thời gian              Số dòng    Churn    Tỷ lệ
------------------------------------------------------------------------
  Train        2024-09-01 → 2025-08-01        62,729   17,214  27.44%
  Validation   2025-09-01 → 2026-02-01        45,586   10,844  23.79%
  Test         2026-03-01 → 2026-06-01        35,336    3,486   9.87%


## 5. Xây dựng Pipeline: Imputer → Scaler → Logistic Regression

**Lưu ý thiết kế:**
- `SimpleImputer(strategy='median')`: Điền giá trị thiếu bằng trung vị (fit chỉ trên Train)
- `StandardScaler()`: Chuẩn hóa Z-score — cần thiết cho Logistic Regression
- `LogisticRegression(penalty='l2', C=1.0, max_iter=1000)`: Mô hình tuyến tính chuẩn
- Không sử dụng `class_weight='balanced'` để giữ baseline thuần túy

In [5]:
# Pipeline: Impute → Scale → LogisticRegression
pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
    ("clf",     LogisticRegression(
        penalty="l2",
        C=1.0,
        max_iter=1000,
        solver="lbfgs",
        random_state=SEED,
        n_jobs=-1,
    )),
])

print("Đang huấn luyện Logistic Regression Baseline...")
print(f"  Số đặc trưng: {len(feature_cols)}")
print(f"  Số mẫu Train: {X_train.shape[0]:,}")

pipe.fit(X_train, y_train)
print("Huấn luyện hoàn tất!")

# Dự đoán xác suất trên cả 3 tập
train_probs = pipe.predict_proba(X_train)[:, 1]
val_probs   = pipe.predict_proba(X_val)[:, 1]
test_probs  = pipe.predict_proba(X_test)[:, 1]

print(f"\nPhạm vi xác suất dự đoán:")
print(f"  Train : [{train_probs.min():.4f}, {train_probs.max():.4f}] | mean = {train_probs.mean():.4f}")
print(f"  Val   : [{val_probs.min():.4f}, {val_probs.max():.4f}] | mean = {val_probs.mean():.4f}")
print(f"  Test  : [{test_probs.min():.4f}, {test_probs.max():.4f}] | mean = {test_probs.mean():.4f}")

Đang huấn luyện Logistic Regression Baseline...
  Số đặc trưng: 379
  Số mẫu Train: 62,729
Huấn luyện hoàn tất!

Phạm vi xác suất dự đoán:
  Train : [0.0000, 0.9838] | mean = 0.2744
  Val   : [0.0000, 0.9998] | mean = 0.2451
  Test  : [0.0000, 0.9988] | mean = 0.1059


## 6. Đánh giá hiệu năng & Phát hiện Data Leakage

In [6]:
def evaluate(name, y_true, probs, threshold=0.5):
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    metrics = {
        "ROC_AUC":   roc_auc_score(y_true, probs),
        "PR_AUC":    average_precision_score(y_true, probs),
        "F1":        f1_score(y_true, preds, zero_division=0),
        "Brier":     brier_score_loss(y_true, probs),
        "TP": tp, "FP": fp, "TN": tn, "FN": fn,
    }
    return metrics

train_m = evaluate("Train", y_train, train_probs)
val_m   = evaluate("Val",   y_val,   val_probs)
test_m  = evaluate("Test",  y_test,  test_probs)

# ── Bảng chỉ số ──────────────────────────────────────────────────
print("=" * 75)
print("BẢNG CHỈ SỐ — LOGISTIC REGRESSION BASELINE")
print("=" * 75)
header = f"  {'Tập':<12} {'ROC-AUC':>10} {'PR-AUC':>10} {'F1':>10} {'Brier':>10}"
print(header)
print("-" * 75)
for name, m in [("Train", train_m), ("Validation", val_m), ("Test", test_m)]:
    print(f"  {name:<12} {m['ROC_AUC']:>10.4f} {m['PR_AUC']:>10.4f} {m['F1']:>10.4f} {m['Brier']:>10.4f}")
print("=" * 75)

# ── Ma trận nhầm lẫn ────────────────────────────────────────────
print(f"\nMa trận nhầm lẫn — Tập TEST (threshold=0.5):")
print(f"  [[TN={test_m['TN']:,}  FP={test_m['FP']:,}]")
print(f"   [FN={test_m['FN']:,}  TP={test_m['TP']:,}]]")

# ══════════════════════════════════════════════════════════════════
#  PHÁT HIỆN DATA LEAKAGE
# ══════════════════════════════════════════════════════════════════
test_auc = test_m["ROC_AUC"]
print("\n" + "=" * 75)
print("KIỂM TRA DATA LEAKAGE")
print("=" * 75)

if test_auc > 0.99:
    print(f"  🔴 CẢNH BÁO NGHIÊM TRỌNG: ROC-AUC = {test_auc:.4f} (> 0.99)")
    print(f"  → Rất có khả năng DATA LEAKAGE trong tập đặc trưng!")
    print(f"  → Hành động: DỪNG LẠI, kiểm tra từng feature trước khi tiếp tục.")
    LEAKAGE_STATUS = "NGHIEM_TRONG"
elif test_auc > 0.95:
    print(f"  🟡 CẢNH BÁO: ROC-AUC = {test_auc:.4f} (0.95 - 0.99)")
    print(f"  → Kết quả cao bất thường cho Logistic Regression.")
    print(f"  → Hành động: Kiểm tra top features có liên quan trực tiếp đến nhãn không.")
    LEAKAGE_STATUS = "CAN_KIEM_TRA"
elif test_auc > 0.80:
    print(f"  🟢 BÌNH THƯỜNG: ROC-AUC = {test_auc:.4f} (0.80 - 0.95)")
    print(f"  → Baseline hợp lý, có thể tiến hành mô hình nâng cao.")
    LEAKAGE_STATUS = "BINH_THUONG"
else:
    print(f"  ⚪ YẾU: ROC-AUC = {test_auc:.4f} (< 0.80)")
    print(f"  → Features chưa đủ mạnh hoặc nhãn cần kiểm tra lại.")
    LEAKAGE_STATUS = "YEU"

print("=" * 75)

BẢNG CHỈ SỐ — LOGISTIC REGRESSION BASELINE
  Tập             ROC-AUC     PR-AUC         F1      Brier
---------------------------------------------------------------------------
  Train            0.8353     0.5652     0.5627     0.1421
  Validation       0.8459     0.5305     0.5491     0.1305
  Test             0.9478     0.5454     0.6337     0.0499

Ma trận nhầm lẫn — Tập TEST (threshold=0.5):
  [[TN=29,913  FP=1,937]
   [FN=971  TP=2,515]]

KIỂM TRA DATA LEAKAGE
  🟢 BÌNH THƯỜNG: ROC-AUC = 0.9478 (0.80 - 0.95)
  → Baseline hợp lý, có thể tiến hành mô hình nâng cao.


## 7. Đồ thị ROC Curve & Precision-Recall Curve

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── ROC Curve ────────────────────────────────────────────────────
ax = axes[0]
for name, y_true, probs, color in [
    ("Train", y_train, train_probs, "#3498db"),
    ("Val",   y_val,   val_probs,   "#e67e22"),
    ("Test",  y_test,  test_probs,  "#e74c3c"),
]:
    fpr, tpr, _ = roc_curve(y_true, probs)
    auc_val = roc_auc_score(y_true, probs)
    ax.plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC={auc_val:.4f})")

ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5, label="Random (AUC=0.5)")
ax.set_xlabel("False Positive Rate", fontsize=11)
ax.set_ylabel("True Positive Rate", fontsize=11)
ax.set_title("ROC Curve — Logistic Regression Baseline", fontsize=12, fontweight="bold")
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.3)

# Vùng cảnh báo leakage
if test_m["ROC_AUC"] > 0.95:
    ax.axhline(y=0.95, color="red", linestyle=":", alpha=0.5, label="Ngưỡng cảnh báo")
    ax.fill_between([0, 0.05], [0.95, 0.95], [1, 1], alpha=0.1, color="red")

# ── Precision-Recall Curve ───────────────────────────────────────
ax = axes[1]
for name, y_true, probs, color in [
    ("Train", y_train, train_probs, "#3498db"),
    ("Val",   y_val,   val_probs,   "#e67e22"),
    ("Test",  y_test,  test_probs,  "#e74c3c"),
]:
    prec, rec, _ = precision_recall_curve(y_true, probs)
    ap = average_precision_score(y_true, probs)
    ax.plot(rec, prec, color=color, lw=2, label=f"{name} (AP={ap:.4f})")

baseline_rate = y_test.mean()
ax.axhline(y=baseline_rate, color="gray", linestyle="--", lw=1, alpha=0.5,
           label=f"Baseline ({baseline_rate:.2%})")
ax.set_xlabel("Recall", fontsize=11)
ax.set_ylabel("Precision", fontsize=11)
ax.set_title("Precision-Recall Curve — Logistic Regression Baseline", fontsize=12, fontweight="bold")
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("output/baseline_logreg_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Đã lưu đồ thị: output/baseline_logreg_curves.png")

Đã lưu đồ thị: output/baseline_logreg_curves.png


## 8. Phân tích hệ số Logistic Regression — Phát hiện Feature đáng ngờ

Logistic Regression cho phép kiểm tra trực tiếp **hệ số (coefficient)** của từng feature.
Nếu một feature có hệ số cực lớn (|coef| >> 1) và tên gợi ý liên quan đến nhãn mục tiêu,
đó là dấu hiệu mạnh mẽ của **data leakage**.

In [8]:
clf = pipe.named_steps["clf"]
coefs = clf.coef_[0]

# Tạo DataFrame hệ số
coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": coefs,
    "abs_coef": np.abs(coefs),
}).sort_values("abs_coef", ascending=False).reset_index(drop=True)

# Top 20 feature có hệ số lớn nhất
print("=" * 75)
print("TOP 20 ĐẶC TRƯNG CÓ HỆ SỐ LỚN NHẤT (|coefficient|)")
print("=" * 75)
print(f"  {'Hạng':>4}  {'Tên đặc trưng':<45} {'Hệ số':>12} {'|Hệ số|':>10}")
print("-" * 75)
for idx, row in coef_df.head(20).iterrows():
    flag = " ⚠️" if row["abs_coef"] > 2.0 else ""
    print(f"  {idx+1:>4}  {row['feature']:<45} {row['coefficient']:>12.6f} {row['abs_coef']:>10.6f}{flag}")
print("=" * 75)

# Cảnh báo nếu có feature hệ số cực lớn
extreme_feats = coef_df[coef_df["abs_coef"] > 3.0]
if len(extreme_feats) > 0:
    print(f"\n⚠️ PHÁT HIỆN {len(extreme_feats)} feature có |coefficient| > 3.0:")
    for _, row in extreme_feats.iterrows():
        print(f"   → {row['feature']} (coef = {row['coefficient']:.6f})")
    print("\n  Các feature này CÓ THỂ chứa thông tin rò rỉ. Hãy kiểm tra kỹ!")

# Lưu CSV
coef_df.to_csv("output/baseline_logreg_coefficients.csv", index=False)
print(f"\nĐã lưu bảng hệ số: output/baseline_logreg_coefficients.csv")

TOP 20 ĐẶC TRƯNG CÓ HỆ SỐ LỚN NHẤT (|coefficient|)
  Hạng  Tên đặc trưng                                        Hệ số    |Hệ số|
---------------------------------------------------------------------------
     1  payment_success_rolling_mean_6m                  -4.231156   4.231156 ⚠️
     2  payment_success_rolling_mean_3m                   3.026816   3.026816 ⚠️
     3  payment_success_lag_1                            -2.898281   2.898281 ⚠️
     4  payment_count_rolling_mean_3m                     2.094795   2.094795 ⚠️
     5  payment_success_slope_3m                          1.975432   1.975432
     6  payment_success_change_1m                        -1.916594   1.916594
     7  payment_success_rolling_sum_3m                   -1.820264   1.820264
     8  orders_lag_1                                      1.693924   1.693924
     9  payment_count_rolling_sum_6m                      1.613664   1.613664
    10  orders_rolling_mean_3m                           -1.601755   1.601755
   

## 9. Biểu đồ Top 25 Feature Coefficients

In [9]:
top_n = 25
top_df = coef_df.head(top_n).copy()

fig, ax = plt.subplots(figsize=(10, 8))
colors = ["#e74c3c" if c > 0 else "#3498db" for c in top_df["coefficient"]]
bars = ax.barh(range(top_n), top_df["coefficient"].values, color=colors, edgecolor="white", height=0.7)
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_df["feature"].values, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("Logistic Regression Coefficient", fontsize=11)
ax.set_title(f"Top {top_n} Feature Coefficients — Baseline Logistic Regression", fontsize=12, fontweight="bold")
ax.axvline(x=0, color="black", lw=0.8)
ax.grid(True, axis="x", alpha=0.3)

# Chú thích màu
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#e74c3c", label="Hệ số dương (tăng xác suất Churn)"),
    Patch(facecolor="#3498db", label="Hệ số âm (giảm xác suất Churn)"),
]
ax.legend(handles=legend_elements, loc="lower right", fontsize=9)

plt.tight_layout()
plt.savefig("output/baseline_logreg_top_features.png", dpi=150, bbox_inches="tight")
plt.show()
print("Đã lưu biểu đồ: output/baseline_logreg_top_features.png")

Đã lưu biểu đồ: output/baseline_logreg_top_features.png


## 10. Tối ưu hóa ngưỡng phân loại (Threshold) trên Validation

In [10]:
thresholds = np.linspace(0.01, 0.99, 99)
f1_scores_list = []
best_f1, best_thresh = -1.0, 0.5

for th in thresholds:
    preds = (val_probs >= th).astype(int)
    f1_val = f1_score(y_val, preds, zero_division=0)
    f1_scores_list.append(f1_val)
    if f1_val > best_f1:
        best_f1, best_thresh = f1_val, float(th)

print(f"Ngưỡng tối ưu (Val F1): {best_thresh:.2f}  |  F1 = {best_f1:.4f}")

# Đánh giá lại Test với ngưỡng tối ưu
test_preds_opt = (test_probs >= best_thresh).astype(int)
test_f1_opt = f1_score(y_test, test_preds_opt, zero_division=0)
tn, fp, fn, tp = confusion_matrix(y_test, test_preds_opt).ravel()

print(f"\nKết quả trên Test (threshold={best_thresh:.2f}):")
print(f"  F1        : {test_f1_opt:.4f}")
print(f"  ROC-AUC   : {test_m['ROC_AUC']:.4f}")
print(f"  PR-AUC    : {test_m['PR_AUC']:.4f}")
print(f"  Confusion : [[TN={tn:,}  FP={fp:,}]  [FN={fn:,}  TP={tp:,}]]")

# Đồ thị F1 vs Threshold
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, f1_scores_list, color="#2ecc71", lw=2)
ax.axvline(x=best_thresh, color="#e74c3c", linestyle="--", lw=1.5, label=f"Optimal: {best_thresh:.2f}")
ax.set_xlabel("Threshold", fontsize=11)
ax.set_ylabel("F1 Score", fontsize=11)
ax.set_title("F1 Score vs Threshold — Validation Set", fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("output/baseline_logreg_threshold.png", dpi=150, bbox_inches="tight")
plt.show()

Ngưỡng tối ưu (Val F1): 0.29  |  F1 = 0.6207

Kết quả trên Test (threshold=0.29):
  F1        : 0.6854
  ROC-AUC   : 0.9478
  PR-AUC    : 0.5454
  Confusion : [[TN=29,233  FP=2,617]  [FN=304  TP=3,182]]


## 11. Phân tích khoảng cách Train-Val-Test (Gap Analysis)

Nếu có **gap lớn** giữa Train AUC và Test AUC → mô hình bị overfitting.
Nếu **tất cả đều cao bất thường** (> 0.99) → data leakage.

In [11]:
gap_train_test = train_m["ROC_AUC"] - test_m["ROC_AUC"]
gap_train_val  = train_m["ROC_AUC"] - val_m["ROC_AUC"]

print("=" * 60)
print("PHÂN TÍCH KHOẢNG CÁCH (GAP ANALYSIS)")
print("=" * 60)
print(f"  Train ROC-AUC  : {train_m['ROC_AUC']:.4f}")
print(f"  Val ROC-AUC    : {val_m['ROC_AUC']:.4f}")
print(f"  Test ROC-AUC   : {test_m['ROC_AUC']:.4f}")
print(f"  Gap Train→Val  : {gap_train_val:+.4f}")
print(f"  Gap Train→Test : {gap_train_test:+.4f}")
print("-" * 60)

if abs(gap_train_test) < 0.02 and train_m["ROC_AUC"] > 0.99:
    print("  📊 Kết quả: Train ≈ Val ≈ Test > 0.99")
    print("  🔴 → Tất cả quá cao + không có gap → DATA LEAKAGE rất có thể!")
elif abs(gap_train_test) > 0.05:
    print(f"  📊 Kết quả: Gap = {gap_train_test:+.4f} (> 0.05)")
    print("  🟡 → Có overfitting, nhưng không phải leakage nếu Test AUC hợp lý")
else:
    print(f"  📊 Kết quả: Gap = {gap_train_test:+.4f}")
    print("  🟢 → Mô hình ổn định, không có dấu hiệu bất thường")
print("=" * 60)

PHÂN TÍCH KHOẢNG CÁCH (GAP ANALYSIS)
  Train ROC-AUC  : 0.8353
  Val ROC-AUC    : 0.8459
  Test ROC-AUC   : 0.9478
  Gap Train→Val  : -0.0105
  Gap Train→Test : -0.1124
------------------------------------------------------------
  📊 Kết quả: Gap = -0.1124 (> 0.05)
  🟡 → Có overfitting, nhưng không phải leakage nếu Test AUC hợp lý


## 12. Phân tích ROC-AUC theo từng Snapshot tháng

Kiểm tra xem hiệu năng mô hình có **ổn định theo thời gian** hay không.
Nếu AUC đột ngột tăng vọt ở một số tháng → có thể leakage cục bộ.

In [12]:
# Dự đoán xác suất cho toàn bộ dataset
all_probs = pipe.predict_proba(df[feature_cols])[:, 1]

# Tính AUC theo từng snapshot
snapshot_auc = []
for snap, grp in df.groupby("snapshot_date"):
    y_snap = grp["churn_next_30d"].values
    p_snap = all_probs[grp.index]
    if len(np.unique(y_snap)) < 2:
        continue
    auc_snap = roc_auc_score(y_snap, p_snap)
    snapshot_auc.append({"snapshot_date": snap, "ROC_AUC": auc_snap, "n_samples": len(grp), "churn_rate": y_snap.mean()})

snap_df = pd.DataFrame(snapshot_auc)

# Đồ thị
fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.plot(snap_df["snapshot_date"], snap_df["ROC_AUC"], "o-", color="#2c3e50", lw=2, markersize=5, label="ROC-AUC")
ax1.axhline(y=0.99, color="red", linestyle=":", lw=1.5, alpha=0.7, label="Ngưỡng leakage (0.99)")
ax1.axhline(y=0.95, color="orange", linestyle=":", lw=1.2, alpha=0.5, label="Ngưỡng cảnh báo (0.95)")

# Đánh dấu vùng Train/Val/Test
ax1.axvspan(TRAIN_START, TRAIN_END, alpha=0.05, color="blue", label="Train")
ax1.axvspan(VAL_START, VAL_END, alpha=0.05, color="orange", label="Val")
ax1.axvspan(TEST_START, TEST_END, alpha=0.05, color="red", label="Test")

ax1.set_xlabel("Snapshot Month", fontsize=11)
ax1.set_ylabel("ROC-AUC", fontsize=11)
ax1.set_title("ROC-AUC theo từng Snapshot tháng — Logistic Regression Baseline", fontsize=12, fontweight="bold")
ax1.legend(loc="lower left", fontsize=8, ncol=3)
ax1.grid(True, alpha=0.3)
ax1.set_ylim([max(0.5, snap_df["ROC_AUC"].min() - 0.05), 1.01])

plt.tight_layout()
plt.savefig("output/baseline_logreg_snapshot_auc.png", dpi=150, bbox_inches="tight")
plt.show()

print("AUC theo snapshot (trích lọc):")
print(snap_df[["snapshot_date", "ROC_AUC", "n_samples", "churn_rate"]].to_string(index=False))

AUC theo snapshot (trích lọc):
snapshot_date  ROC_AUC  n_samples  churn_rate
   2023-08-01 0.472222         25    0.360000
   2023-09-01 0.610040        323    0.204334
   2023-10-01 0.702919        598    0.244147
   2023-11-01 0.731205        922    0.246204
   2023-12-01 0.743795       1223    0.268193
   2024-01-01 0.783838       1505    0.272425
   2024-02-01 0.791040       1789    0.247065
   2024-03-01 0.783198       2028    0.265286
   2024-04-01 0.793209       2333    0.271324
   2024-05-01 0.799895       2623    0.267251
   2024-06-01 0.803584       2888    0.261773
   2024-07-01 0.802379       3141    0.270614
   2024-08-01 0.809565       3418    0.279403
   2024-09-01 0.817749       3711    0.273511
   2024-10-01 0.831127       3964    0.272957
   2024-11-01 0.828203       4305    0.270151
   2024-12-01 0.829617       4543    0.277130
   2025-01-01 0.831582       4831    0.270337
   2025-02-01 0.826991       5111    0.275680
   2025-03-01 0.837963       5357    0.275154
   

## 13. Kết luận & Khuyến nghị

In [13]:
print("\n" + "=" * 75)
print("KẾT LUẬN — BASELINE LOGISTIC REGRESSION")
print("=" * 75)
print(f"\n  Mô hình          : Logistic Regression (L2, C=1.0)")
print(f"  Số đặc trưng     : {len(feature_cols)}")
print(f"  Ngưỡng tối ưu    : {best_thresh:.2f}")
print()
print(f"  ┌─────────────────────────────────────────────────┐")
print(f"  │  TEST SET RESULTS                               │")
print(f"  │  ROC-AUC  : {test_m['ROC_AUC']:.4f}                            │")
print(f"  │  PR-AUC   : {test_m['PR_AUC']:.4f}                            │")
print(f"  │  F1 Score : {test_f1_opt:.4f} (threshold={best_thresh:.2f})          │")
print(f"  │  Brier    : {test_m['Brier']:.4f}                            │")
print(f"  └─────────────────────────────────────────────────┘")
print()

if LEAKAGE_STATUS == "NGHIEM_TRONG":
    print("  ╔═════════════════════════════════════════════════════════════╗")
    print("  ║  🔴 PHÁN QUYẾT: DATA LEAKAGE NGHIÊM TRỌNG               ║")
    print("  ║                                                           ║")
    print("  ║  → DỪNG LẠI NGAY. KHÔNG huấn luyện mô hình nâng cao.    ║")
    print("  ║  → Kiểm tra lại Feature Engineering:                     ║")
    print("  ║    1. Loại bỏ các feature có |coef| > 3.0                ║")
    print("  ║    2. Kiểm tra feature có chứa thông tin tương lai?      ║")
    print("  ║    3. Kiểm tra cột rule1/rule2/rule3 có bị lẫn vào?     ║")
    print("  ║    4. Chạy lại baseline sau khi loại bỏ feature nghi ngờ ║")
    print("  ╚═════════════════════════════════════════════════════════════╝")
elif LEAKAGE_STATUS == "CAN_KIEM_TRA":
    print("  ╔═════════════════════════════════════════════════════════════╗")
    print("  ║  🟡 PHÁN QUYẾT: CẦN KIỂM TRA KỸ                        ║")
    print("  ║                                                           ║")
    print("  ║  → AUC cao bất thường cho mô hình tuyến tính.            ║")
    print("  ║  → Kiểm tra top features ở Cell 8 có hợp lý không.       ║")
    print("  ║  → Nếu xác nhận OK → có thể tiến hành mô hình nâng cao. ║")
    print("  ╚═════════════════════════════════════════════════════════════╝")
elif LEAKAGE_STATUS == "BINH_THUONG":
    print("  ╔═════════════════════════════════════════════════════════════╗")
    print("  ║  🟢 PHÁN QUYẾT: BASELINE HỢP LỆ — KHÔNG CÓ LEAKAGE     ║")
    print("  ║                                                           ║")
    print("  ║  → Tiến hành huấn luyện mô hình nâng cao:                ║")
    print("  ║    1. LightGBM (Mô hình chính)                           ║")
    print("  ║    2. XGBoost (Ensemble partner)                          ║")
    print("  ║    3. Stacking/Blending LightGBM + XGBoost + LogReg      ║")
    print("  ╚═════════════════════════════════════════════════════════════╝")
else:
    print("  ╔═════════════════════════════════════════════════════════════╗")
    print("  ║  ⚪ PHÁN QUYẾT: BASELINE YẾU                             ║")
    print("  ║                                                           ║")
    print("  ║  → Cần bổ sung thêm features hoặc kiểm tra lại nhãn.    ║")
    print("  ║  → Mô hình phi tuyến (LightGBM) có thể giúp cải thiện.  ║")
    print("  ╚═════════════════════════════════════════════════════════════╝")

print("\n" + "=" * 75)


KẾT LUẬN — BASELINE LOGISTIC REGRESSION

  Mô hình          : Logistic Regression (L2, C=1.0)
  Số đặc trưng     : 379
  Ngưỡng tối ưu    : 0.29

  ┌─────────────────────────────────────────────────┐
  │  TEST SET RESULTS                               │
  │  ROC-AUC  : 0.9478                            │
  │  PR-AUC   : 0.5454                            │
  │  F1 Score : 0.6854 (threshold=0.29)          │
  │  Brier    : 0.0499                            │
  └─────────────────────────────────────────────────┘

  ╔═════════════════════════════════════════════════════════════╗
  ║  🟢 PHÁN QUYẾT: BASELINE HỢP LỆ — KHÔNG CÓ LEAKAGE     ║
  ║                                                           ║
  ║  → Tiến hành huấn luyện mô hình nâng cao:                ║
  ║    1. LightGBM (Mô hình chính)                           ║
  ║    2. XGBoost (Ensemble partner)                          ║
  ║    3. Stacking/Blending LightGBM + XGBoost + LogReg      ║
  ╚═════════════════════════════════════